# 04 - 自定义训练循环

## 学习目标
- 掌握 tf.GradientTape
- 实现自定义训练循环
- 自定义损失函数

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

## 1. 基本自定义训练

In [ ]:
# 准备数据
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 创建 Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(32)

# 构建模型
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10)
])

# 定义优化器和损失函数
optimizer = keras.optimizers.Adam()
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [ ]:
# 自定义训练循环
@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:
        logits = model(images, training=True)
        loss = loss_fn(labels, logits)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

# 训练
epochs = 5
for epoch in range(epochs):
    total_loss = 0
    steps = 0
    
    for images, labels in train_dataset:
        loss = train_step(images, labels)
        total_loss += loss
        steps += 1
    
    avg_loss = total_loss / steps
    print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}')

## 2. 自定义损失函数

In [ ]:
# Huber Loss
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    quadratic = tf.minimum(abs_error, delta)
    linear = abs_error - quadratic
    return 0.5 * quadratic**2 + delta * linear

# Focal Loss
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    y_pred = tf.nn.softmax(y_pred)
    cross_entropy = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    p_t = tf.exp(-cross_entropy)
    loss = alpha * ((1 - p_t) ** gamma) * cross_entropy
    return tf.reduce_mean(loss)

print('自定义损失函数定义完成')

## 3. 自定义模型

In [ ]:
class CustomModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.conv1 = layers.Conv2D(32, 3, activation='relu')
        self.pool = layers.MaxPooling2D()
        self.flatten = layers.Flatten()
        self.dense1 = layers.Dense(128, activation='relu')
        self.dense2 = layers.Dense(10)
        self.dropout = layers.Dropout(0.5)
    
    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.dense1(x)
        if training:
            x = self.dropout(x, training=True)
        return self.dense2(x)

# 使用自定义模型
custom_model = CustomModel()
output = custom_model(tf.random.normal([1, 28, 28, 1]))
print(f'输出形状: {output.shape}')

## 小结
- GradientTape 实现自动微分
- 自定义训练循环提供最大灵活性
- 可以自定义损失函数和模型